# 03 — Data Enrichment: Market Data, Technicals & News

This notebook enriches your portfolio holdings with:
- **Current & historical prices** (yfinance)
- **Technical indicators** — RSI, SMA 50/200, EMA 12/26, MACD, Bollinger Bands (pandas-ta)
- **Fundamentals** — P/E, market cap, dividend yield, 52-week range, sector (yfinance)
- **Recent news headlines** (yfinance)

The output is an enriched JSON file ready to feed into Claude for analysis in Phase 3.

**Prerequisites:**
- Run notebooks 01 and 02 first
- `portfolio_snapshot.json` must exist
- Run `uv sync` after pulling this update (new dependencies: yfinance, pandas-ta)

## Setup

In [1]:
import json
import os
import warnings
from datetime import datetime, timedelta

import pandas as pd
import pandas_ta as ta
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Load portfolio snapshot from notebook 02
SNAPSHOT_FILE = os.path.join("..", "portfolio_snapshot.json")

if not os.path.exists(SNAPSHOT_FILE):
    raise FileNotFoundError(
        "portfolio_snapshot.json not found. Run notebook 02 first."
    )

with open(SNAPSHOT_FILE, "r") as f:
    portfolio = json.load(f)

holdings_df = pd.DataFrame(portfolio["holdings"])

# Get unique tickers (exclude cash and None)
tickers = (
    holdings_df[holdings_df["type"] != "cash"]["ticker"]
    .dropna()
    .unique()
    .tolist()
)

# Add RSU tickers for price tracking even if not currently held
RSU_TICKERS = ["SNOW"]
for t in RSU_TICKERS:
    if t not in tickers:
        tickers.append(t)
        print(f"📌 Added RSU ticker {t} for price tracking")

print(f"✅ Loaded portfolio: {len(holdings_df)} positions")
print(f"📊 Unique tickers to enrich: {len(tickers)}")
print(f"   {tickers}")

📌 Added RSU ticker SNOW for price tracking
✅ Loaded portfolio: 70 positions
📊 Unique tickers to enrich: 52
   ['AOR', 'JEPQ', 'SKYY', 'JEPI', 'VGT', 'SUSA', 'SOFI', 'MSFT', 'VOO', 'BRK.B', 'AAPL', 'VTI', 'NVDA', 'IXUS', 'V', 'SPY', 'GLTR', 'SFY', 'QQQ', 'AMZN', 'SHLD', 'LQD', 'VOX', 'MA', 'VNQ', 'GAMR', 'IEFA', 'IJH', 'VWO', 'DVN', 'INDA', 'VEA', 'VWOB', 'IVW', 'DYNF', 'ARKQ', 'IVE', 'VB', 'GURU', 'IJR', 'AGG', 'VPL', 'KR', 'DIS', 'THRO', 'BAI', 'NFLX', 'HLN', 'R', 'EQIX', 'GSK', 'SNOW']


## 1. Fetch Historical Prices (6 months daily)

In [2]:
LOOKBACK_DAYS = 200  # enough for SMA-200
start_date = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")

print(f"Downloading daily prices since {start_date}...\n")

price_data = {}  # ticker -> DataFrame
failed_tickers = []

for ticker in tickers:
    try:
        df = yf.download(ticker, start=start_date, progress=False, auto_adjust=True)
        if df.empty:
            print(f"  ⚠️  {ticker}: No data returned (may be delisted or invalid)")
            failed_tickers.append(ticker)
        else:
            # Flatten multi-level columns if present
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            price_data[ticker] = df
            print(f"  ✅ {ticker}: {len(df)} days, latest close ${df['Close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")
        failed_tickers.append(ticker)

print(f"\n📈 Price data fetched for {len(price_data)}/{len(tickers)} tickers")
if failed_tickers:
    print(f"⚠️  Failed/skipped: {failed_tickers}")

  ✅ AOR: 139 days, latest close $65.47


  ✅ JEPQ: 139 days, latest close $56.85


  ✅ SKYY: 139 days, latest close $113.17


  ✅ JEPI: 139 days, latest close $58.09


  ✅ VGT: 139 days, latest close $719.03
  ✅ SUSA: 139 days, latest close $136.73


...

  ✅ THRO: 139 days, latest close $37.41
  ✅ BAI: 139 days, latest close $32.70


  ✅ NFLX: 139 days, latest close $99.02
  ✅ HLN: 139 days, latest close $10.28


  ✅ R: 139 days, latest close $198.88
  ✅ EQIX: 139 days, latest close $937.20


  ✅ GSK: 139 days, latest close $54.51


  ✅ SNOW: 139 days, latest close $180.48

📈 Price data fetched for 51/52 tickers
⚠️  Failed/skipped: ['BRK.B']


## 2. Calculate Technical Indicators

For each ticker, we compute:
- **SMA 50 / SMA 200** — trend (Golden Cross / Death Cross)
- **EMA 12 / EMA 26** — momentum
- **RSI (14)** — overbought (>70) / oversold (<30)
- **MACD** — trend direction and momentum
- **Bollinger Bands (20, 2)** — volatility and mean reversion

In [3]:
technicals = {}  # ticker -> dict of latest indicator values

for ticker, df in price_data.items():
    close = df["Close"]
    
    # Moving averages
    sma_50 = ta.sma(close, length=50)
    sma_200 = ta.sma(close, length=200)
    ema_12 = ta.ema(close, length=12)
    ema_26 = ta.ema(close, length=26)
    
    # RSI
    rsi = ta.rsi(close, length=14)
    
    # MACD
    macd_df = ta.macd(close, fast=12, slow=26, signal=9)
    
    # Bollinger Bands
    bbands = ta.bbands(close, length=20, std=2)
    
    latest_price = close.iloc[-1]
    
    tech = {
        "price": round(float(latest_price), 2),
        "sma_50": round(float(sma_50.iloc[-1]), 2) if sma_50 is not None and not sma_50.empty and pd.notna(sma_50.iloc[-1]) else None,
        "sma_200": round(float(sma_200.iloc[-1]), 2) if sma_200 is not None and not sma_200.empty and pd.notna(sma_200.iloc[-1]) else None,
        "ema_12": round(float(ema_12.iloc[-1]), 2) if ema_12 is not None and not ema_12.empty else None,
        "ema_26": round(float(ema_26.iloc[-1]), 2) if ema_26 is not None and not ema_26.empty else None,
        "rsi_14": round(float(rsi.iloc[-1]), 2) if rsi is not None and not rsi.empty and pd.notna(rsi.iloc[-1]) else None,
    }
    
    # MACD values
    if macd_df is not None and not macd_df.empty:
        cols = macd_df.columns
        tech["macd"] = round(float(macd_df[cols[0]].iloc[-1]), 4) if pd.notna(macd_df[cols[0]].iloc[-1]) else None
        tech["macd_signal"] = round(float(macd_df[cols[2]].iloc[-1]), 4) if pd.notna(macd_df[cols[2]].iloc[-1]) else None
        tech["macd_hist"] = round(float(macd_df[cols[1]].iloc[-1]), 4) if pd.notna(macd_df[cols[1]].iloc[-1]) else None
    
    # Bollinger Bands
    if bbands is not None and not bbands.empty:
        bb_cols = bbands.columns
        tech["bb_lower"] = round(float(bbands[bb_cols[0]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[0]].iloc[-1]) else None
        tech["bb_mid"] = round(float(bbands[bb_cols[1]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[1]].iloc[-1]) else None
        tech["bb_upper"] = round(float(bbands[bb_cols[2]].iloc[-1]), 2) if pd.notna(bbands[bb_cols[2]].iloc[-1]) else None
    
    # Derived signals
    if tech["sma_50"] and tech["sma_200"]:
        tech["golden_cross"] = tech["sma_50"] > tech["sma_200"]
    if tech["rsi_14"]:
        tech["rsi_signal"] = (
            "overbought" if tech["rsi_14"] > 70
            else "oversold" if tech["rsi_14"] < 30
            else "neutral"
        )
    if tech.get("bb_lower") and tech.get("bb_upper"):
        tech["bb_position"] = (
            "below_lower" if latest_price < tech["bb_lower"]
            else "above_upper" if latest_price > tech["bb_upper"]
            else "within_bands"
        )
    
    # Price vs moving averages
    if tech["sma_50"]:
        tech["pct_above_sma50"] = round((latest_price / tech["sma_50"] - 1) * 100, 2)
    if tech["sma_200"]:
        tech["pct_above_sma200"] = round((latest_price / tech["sma_200"] - 1) * 100, 2)
    
    technicals[ticker] = tech

print(f"✅ Technicals computed for {len(technicals)} tickers")

✅ Technicals computed for 51 tickers


### Technicals Overview Table

In [4]:
tech_df = pd.DataFrame.from_dict(technicals, orient="index")
tech_df.index.name = "ticker"

# Show key columns
display_cols = ["price", "sma_50", "sma_200", "rsi_14", "rsi_signal", "macd_hist", "bb_position"]
available_cols = [c for c in display_cols if c in tech_df.columns]

tech_df[available_cols]

,price,sma_50,sma_200,rsi_14,rsi_signal,macd_hist,bb_position
ticker,,,,,,,
AOR,65.47,66.31,None,36.42,neutral,-0.2217,below_lower
JEPQ,56.85,57.96,None,43.04,neutral,-0.0105,within_bands
SKYY,113.17,119.95,None,48.01,neutral,0.8616,within_bands
JEPI,58.09,58.21,None,41.67,neutral,-0.1317,below_lower
VGT,719.03,747.76,None,41.92,neutral,-0.5615,within_bands
SUSA,136.73,140.58,None,37.59,neutral,-0.2827,below_lower
SOFI,18.90,23.15,None,39.17,neutral,0.2436,within_bands
MSFT,408.96,436.97,None,47.58,neutral,4.1437,within_bands
VOO,618.43,632.78,None,38.52,neutral,-1.1065,below_lower


## 3. Fetch Fundamentals

In [5]:
fundamentals = {}  # ticker -> dict

for ticker in price_data.keys():
    try:
        info = yf.Ticker(ticker).info
        
        fundamentals[ticker] = {
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "market_cap": info.get("marketCap"),
            "pe_trailing": info.get("trailingPE"),
            "pe_forward": info.get("forwardPE"),
            "peg_ratio": info.get("pegRatio"),
            "price_to_book": info.get("priceToBook"),
            "dividend_yield": info.get("dividendYield"),
            "beta": info.get("beta"),
            "fifty_two_week_high": info.get("fiftyTwoWeekHigh"),
            "fifty_two_week_low": info.get("fiftyTwoWeekLow"),
            "avg_volume": info.get("averageVolume"),
            "revenue_growth": info.get("revenueGrowth"),
            "earnings_growth": info.get("earningsGrowth"),
            "profit_margin": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "debt_to_equity": info.get("debtToEquity"),
            "free_cash_flow": info.get("freeCashflow"),
            "analyst_target": info.get("targetMeanPrice"),
            "analyst_rec": info.get("recommendationKey"),
            "short_description": info.get("longBusinessSummary", "")[:200],
        }
        print(f"  ✅ {ticker}: {fundamentals[ticker].get('sector', 'N/A')} | P/E {fundamentals[ticker].get('pe_trailing', 'N/A')}")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")

print(f"\n✅ Fundamentals fetched for {len(fundamentals)}/{len(price_data)} tickers")

  ✅ AOR: None | P/E 22.032423


  ✅ JEPQ: None | P/E 32.88357
  ✅ SKYY: None | P/E 32.23422


  ✅ JEPI: None | P/E 27.084003
  ✅ VGT: None | P/E 33.48565


  ✅ SUSA: None | P/E 26.038519


  ✅ SOFI: Financial Services | P/E 48.46154


...

  ✅ THRO: None | P/E 26.452568
  ✅ BAI: None | P/E 35.100044


  ✅ NFLX: Communication Services | P/E 39.13834
  ✅ HLN: Healthcare | P/E 20.979591


  ✅ R: Industrials | P/E 16.587156


  ✅ EQIX: Real Estate | P/E 68.30904
  ✅ GSK: Healthcare | P/E 14.692721


  ✅ SNOW: Technology | P/E None

✅ Fundamentals fetched for 51/51 tickers


In [6]:
fund_df = pd.DataFrame.from_dict(fundamentals, orient="index")
fund_df.index.name = "ticker"

display_cols = ["sector", "market_cap", "pe_trailing", "pe_forward", "dividend_yield", "beta", "analyst_rec", "analyst_target"]
available_cols = [c for c in display_cols if c in fund_df.columns]

fund_df[available_cols]

,sector,market_cap,pe_trailing,pe_forward,dividend_yield,beta,analyst_rec,analyst_target
ticker,,,,,,,,
AOR,NaN,NaN,22.032423,NaN,2.46,NaN,NaN,NaN
JEPQ,NaN,NaN,32.883570,NaN,10.58,NaN,NaN,NaN
SKYY,NaN,NaN,32.234220,NaN,0.00,NaN,NaN,NaN
JEPI,NaN,NaN,27.084003,NaN,7.91,NaN,NaN,NaN
VGT,NaN,NaN,33.485650,NaN,0.42,NaN,NaN,NaN
SUSA,NaN,NaN,26.038519,NaN,0.89,NaN,NaN,NaN
SOFI,Financial Services,2.410249e+10,48.461540,23.935865,NaN,2.259,hold,26.50000
MSFT,Technology,3.039545e+12,25.591990,21.704426,0.89,1.108,strong_buy,595.99567
VOO,NaN,NaN,27.106470,NaN,1.12,NaN,NaN,NaN


## 4. Fetch Recent News

In [7]:
MAX_NEWS_PER_TICKER = 5

news_data = {}  # ticker -> list of headlines

for ticker in price_data.keys():
    try:
        yf_ticker = yf.Ticker(ticker)
        raw_news = yf_ticker.news or []
        
        headlines = []
        for item in raw_news[:MAX_NEWS_PER_TICKER]:
            headline = {
                "title": item.get("title", ""),
                "publisher": item.get("publisher", ""),
                "link": item.get("link", ""),
            }
            # Extract publish time if available
            pub_time = item.get("providerPublishTime")
            if pub_time:
                headline["published"] = datetime.fromtimestamp(pub_time).strftime("%Y-%m-%d %H:%M")
            headlines.append(headline)
        
        news_data[ticker] = headlines
        print(f"  ✅ {ticker}: {len(headlines)} articles")
    except Exception as e:
        print(f"  ⚠️  {ticker}: {e}")
        news_data[ticker] = []

print(f"\n📰 News fetched for {len(news_data)} tickers")

  ✅ AOR: 5 articles
  ✅ JEPQ: 5 articles
  ✅ SKYY: 2 articles


  ✅ JEPI: 5 articles
  ✅ VGT: 5 articles


  ✅ SUSA: 1 articles
  ✅ SOFI: 5 articles


  ✅ MSFT: 5 articles


  ✅ VOO: 5 articles
  ✅ AAPL: 5 articles


...

  ✅ VPL: 0 articles
  ✅ KR: 5 articles


  ✅ DIS: 5 articles
  ✅ THRO: 2 articles
  ✅ BAI: 5 articles


  ✅ NFLX: 5 articles
  ✅ HLN: 5 articles


  ✅ R: 5 articles
  ✅ EQIX: 5 articles


  ✅ GSK: 5 articles
  ✅ SNOW: 5 articles

📰 News fetched for 51 tickers


In [8]:
# Preview news for first ticker
preview_ticker = list(news_data.keys())[0] if news_data else None

if preview_ticker and news_data[preview_ticker]:
    print(f"\n📰 Recent headlines for {preview_ticker}:")
    for article in news_data[preview_ticker]:
        date = article.get('published', 'N/A')
        print(f"  [{date}] {article['title']}")
        print(f"           — {article['publisher']}")
else:
    print("No news available to preview.")


📰 Recent headlines for AOR:
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 
  [N/A] 
           — 


## 5. Performance Metrics

In [9]:
performance = {}  # ticker -> return periods

for ticker, df in price_data.items():
    close = df["Close"]
    latest = close.iloc[-1]
    
    perf = {}
    
    # 1-day return
    if len(close) >= 2:
        perf["return_1d"] = round(float((latest / close.iloc[-2] - 1) * 100), 2)
    
    # 1-week return
    if len(close) >= 5:
        perf["return_1w"] = round(float((latest / close.iloc[-5] - 1) * 100), 2)
    
    # 1-month return
    if len(close) >= 21:
        perf["return_1m"] = round(float((latest / close.iloc[-21] - 1) * 100), 2)
    
    # 3-month return
    if len(close) >= 63:
        perf["return_3m"] = round(float((latest / close.iloc[-63] - 1) * 100), 2)
    
    # 6-month return
    if len(close) >= 126:
        perf["return_6m"] = round(float((latest / close.iloc[-126] - 1) * 100), 2)
    
    # 52-week high/low proximity
    high = fundamentals.get(ticker, {}).get("fifty_two_week_high")
    low = fundamentals.get(ticker, {}).get("fifty_two_week_low")
    if high:
        perf["pct_from_52w_high"] = round(float((latest / high - 1) * 100), 2)
    if low:
        perf["pct_from_52w_low"] = round(float((latest / low - 1) * 100), 2)
    
    # Analyst target upside/downside
    target = fundamentals.get(ticker, {}).get("analyst_target")
    if target:
        perf["analyst_upside_pct"] = round(float((target / latest - 1) * 100), 2)
    
    performance[ticker] = perf

perf_df = pd.DataFrame.from_dict(performance, orient="index")
perf_df.index.name = "ticker"

print("📊 Performance Summary:")
perf_df

📊 Performance Summary:


,return_1d,return_1w,return_1m,return_3m,return_6m,pct_from_52w_high,pct_from_52w_low,analyst_upside_pct
ticker,,,,,,,,
AOR,-0.73,-2.31,-0.58,1.17,5.06,-3.31,23.60,NaN
JEPQ,-1.37,-1.27,0.74,-1.18,6.49,-5.47,28.30,NaN
SKYY,-0.29,2.49,4.78,-13.78,-12.41,-21.27,32.55,NaN
JEPI,-0.90,-2.17,0.33,2.45,5.65,-3.02,16.32,NaN
VGT,-1.94,-2.02,1.48,-5.87,3.12,-10.90,59.43,NaN
SUSA,-1.26,-2.31,-0.64,-2.31,4.03,-4.50,37.44,NaN
SOFI,-1.82,2.77,-2.88,-36.15,-26.17,-42.25,119.77,40.21
MSFT,-0.42,2.61,4.12,-14.75,-17.04,-26.37,18.61,45.73
VOO,-1.34,-2.04,-0.75,-1.45,4.54,-3.64,39.66,NaN


## 6. Export Enriched Portfolio for Claude (Phase 3)

We combine everything into a single JSON file:
- Portfolio holdings (from Plaid)
- Technicals per ticker
- Fundamentals per ticker
- Performance metrics per ticker
- Recent news per ticker

In [10]:
enriched_portfolio = {
    "generated_at": datetime.now().isoformat(),
    "summary": portfolio["summary"],
    "holdings": portfolio["holdings"],
    "enrichment": {},
}

# Combine all enrichment data per ticker
for ticker in price_data.keys():
    enriched_portfolio["enrichment"][ticker] = {
        "technicals": technicals.get(ticker, {}),
        "fundamentals": fundamentals.get(ticker, {}),
        "performance": performance.get(ticker, {}),
        "news": news_data.get(ticker, []),
    }

# Save
export_path = os.path.join("..", "enriched_portfolio.json")
with open(export_path, "w") as f:
    json.dump(enriched_portfolio, f, indent=2, default=str)

# File size
size_kb = os.path.getsize(export_path) / 1024

print(f"💾 Enriched portfolio saved to {export_path} ({size_kb:.1f} KB)")
print(f"")
print(f"   📋 Holdings:     {len(enriched_portfolio['holdings'])} positions")
print(f"   📈 Enriched:     {len(enriched_portfolio['enrichment'])} tickers")
print(f"   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands")
print(f"   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins")
print(f"   📰 News:         {sum(len(v) for v in news_data.values())} total articles")
print(f"")
print(f"👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).")

💾 Enriched portfolio saved to ../enriched_portfolio.json (138.2 KB)

   📋 Holdings:     70 positions
   📈 Enriched:     51 tickers
   🔧 Technicals:   RSI, SMA 50/200, EMA, MACD, Bollinger Bands
   📊 Fundamentals: P/E, market cap, beta, analyst targets, margins
   📰 News:         225 total articles

👉 This file will be fed to Claude for analysis in notebook 04 (Phase 3).


## Quick Flags (Preview of What Claude Will Analyze)

A quick scan for positions that might warrant attention:

In [11]:
print("🚩 POSITIONS WORTH WATCHING:\n")

for ticker, tech in technicals.items():
    flags = []
    
    # RSI extremes
    if tech.get("rsi_signal") == "overbought":
        flags.append(f"⚠️  RSI {tech['rsi_14']} — overbought")
    elif tech.get("rsi_signal") == "oversold":
        flags.append(f"🟢 RSI {tech['rsi_14']} — oversold (potential buy)")
    
    # Death cross
    if tech.get("golden_cross") is False:
        flags.append("☠️  Death Cross (SMA 50 < SMA 200)")
    
    # Bollinger Band breakout
    if tech.get("bb_position") == "below_lower":
        flags.append("📉 Below lower Bollinger Band")
    elif tech.get("bb_position") == "above_upper":
        flags.append("📈 Above upper Bollinger Band")
    
    # Near 52-week low
    perf = performance.get(ticker, {})
    if perf.get("pct_from_52w_low") is not None and perf["pct_from_52w_low"] < 5:
        flags.append(f"🔻 Near 52-week low ({perf['pct_from_52w_low']:+.1f}%)")
    
    # Near 52-week high
    if perf.get("pct_from_52w_high") is not None and perf["pct_from_52w_high"] > -5:
        flags.append(f"🔺 Near 52-week high ({perf['pct_from_52w_high']:+.1f}%)")
    
    # Analyst upside/downside
    if perf.get("analyst_upside_pct") is not None:
        if perf["analyst_upside_pct"] > 20:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% upside")
        elif perf["analyst_upside_pct"] < -10:
            flags.append(f"📊 Analyst target: {perf['analyst_upside_pct']:+.1f}% downside")
    
    if flags:
        print(f"  {ticker}:")
        for flag in flags:
            print(f"    {flag}")
        print()

print("\n💡 Claude will analyze these signals in context with fundamentals and news in Phase 3.")

🚩 POSITIONS WORTH WATCHING:

  AOR:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-3.3%)

  JEPI:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-3.0%)

  SUSA:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-4.5%)

  SOFI:
    📊 Analyst target: +40.2% upside

  MSFT:
    📊 Analyst target: +45.7% upside

  VOO:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-3.6%)

  VTI:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-3.8%)

  NVDA:
    📊 Analyst target: +49.1% upside

  IXUS:
    📉 Below lower Bollinger Band

  V:
    📊 Analyst target: +26.2% upside

  SPY:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-3.6%)

  SFY:
    📉 Below lower Bollinger Band
    🔺 Near 52-week high (-4.9%)

  AMZN:
    📊 Analyst target: +31.6% upside

  SHLD:
    🔺 Near 52-week high (-1.6%)

  LQD:
    🔺 Near 52-week high (-2.5%)

  MA:
    📊 Analyst target: +26.9% upside

  VNQ:
    🔺 Near 52-week high (-2.8%)

  IEFA:
    📉 Below lower Boll